# Ebene 1: Chronos-2 als Dynamikmodell für Pendulum-v1

Reine Modellbewertung: (1) Ein-Schritt-Genauigkeit, (2) Aktionssensitivität,
(3) nötige Kontextlänge, (4) Mehrschritt-Rollout-Fehler.

Modelle: **Chronos-2** (zero-shot, Aktion als bekannte Zukunfts-Kovariate),
**VARX** (statsmodels, Aktion exogen), **Linear** (kleinste Quadrate).
Zustand $s = (\theta, \dot\theta)$, Daten: 50 Random-Action-Episoden (Seed 0),
Baselines fitten auf Episoden 0-39, evaluiert wird auf 40-49.
Aktions-Alignment: Kovariate zum Zeitpunkt $t$ = Aktion, die $s_t$ erzeugt hat
(siehe `tsfmrl.pendulum`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from pendulum import ChronosDynamics, VARDynamics, LinearDynamics, collect_random_dataset

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)


def make_windows(states, actions, L, H, max_windows=None, seed=0):
    """Fenster (Kontext, Zukunft) aus Episoden schneiden; Alignment wie tsfmrl.pendulum."""
    ctx_s, ctx_a, fut_a, fut_s = [], [], [], []
    for ep in range(len(actions)):
        for t0 in range(1, actions.shape[1] - L - H + 2):
            t1 = t0 + L
            ctx_s.append(states[ep, t0:t1])
            ctx_a.append(actions[ep, t0 - 1 : t1 - 1])
            fut_a.append(actions[ep, t1 - 1 : t1 - 1 + H])
            fut_s.append(states[ep, t1 : t1 + H])
    w = [np.stack(x) for x in (ctx_s, ctx_a, fut_a, fut_s)]
    if max_windows is not None and len(w[0]) > max_windows:
        idx = np.random.default_rng(seed).choice(len(w[0]), size=max_windows, replace=False)
        w = [x[idx] for x in w]
    return w


dataset = collect_random_dataset(n_episodes=50, seed=SEED)
states, actions = dataset["states"], dataset["actions"]

fig, axes = plt.subplots(3, 1, figsize=(8, 5), sharex=True)
for ax, (label, y) in zip(axes, [(r"$\theta$", states[0, :, 0]),
                                 (r"$\dot\theta$", states[0, :, 1]),
                                 ("action", actions[0, :, 0])]):
    ax.plot(y)
    ax.set_ylabel(label)
axes[-1].set_xlabel("step")
fig.tight_layout()

In [ ]:
chronos_model = ChronosDynamics()  # zero-shot, kein Trainings-Split
var_model = VARDynamics(states[:40], actions[:40])
lin_model = LinearDynamics(states[:40], actions[:40])
models = [chronos_model, var_model, lin_model]

print("Chronos:", chronos_model.model_id, "auf", chronos_model.device)
print("VARX Lag (AIC):", var_model.lag)
print("Linear B (analytisch [0, 3·dt/(m·l²)] = [0, 0.15]):", lin_model.B.ravel())

## Test 1: Ein-Schritt-Vorhersage (L=64, ~2000 Fenster, ein Batch-Call pro Modell)

In [ ]:
ctx_s, ctx_a, fut_a, fut_s = make_windows(states[40:], actions[40:], L=64, H=1, max_windows=2000)

preds_1step = {m.name: m.predict(ctx_s, ctx_a, fut_a)[:, 0] for m in models}
true_1step = fut_s[:, 0]

pd.DataFrame({
    name: {"MSE": np.mean((p - true_1step) ** 2), "MAE": np.mean(np.abs(p - true_1step))}
    for name, p in preds_1step.items()
}).T

In [ ]:
fig, axes = plt.subplots(2, len(models), figsize=(12, 7), squeeze=False)
for col, (name, p) in enumerate(preds_1step.items()):
    for row, label in enumerate([r"$\theta$", r"$\dot\theta$"]):
        ax = axes[row][col]
        ax.scatter(true_1step[:, row], p[:, row], s=4, alpha=0.4)
        lo, hi = true_1step[:, row].min(), true_1step[:, row].max()
        ax.plot([lo, hi], [lo, hi], "k--", lw=1)
        ax.set(xlabel=f"wahr {label}", ylabel=f"vorhergesagt {label}")
        if row == 0:
            ax.set_title(name)
fig.tight_layout()

## Test 2: Aktionssensitivität

Gleicher Kontext, kontrafaktische nächste Aktion $a \in \{-2,-1,0,1,2\}$.

ein Modell, das die Aktions-Kovariate ignoriert, produziert flache Linien
(Steigung 0 statt 0.15).

In [ ]:
probe_actions = np.array([-2.0, -1.0, 0.0, 1.0, 2.0], dtype=np.float32)
n_ctx, n_probe = 200, len(probe_actions)
idx = np.random.default_rng(SEED).choice(len(ctx_s), size=n_ctx, replace=False)

rep_s = np.repeat(ctx_s[idx], n_probe, axis=0)          # ein Batch: 200 x 5 Tasks
rep_a = np.repeat(ctx_a[idx], n_probe, axis=0)
rep_f = np.tile(probe_actions.reshape(n_probe, 1, 1), (n_ctx, 1, 1)).astype(np.float32)

probe_preds = {m.name: m.predict(rep_s, rep_a, rep_f).reshape(n_ctx, n_probe, 2) for m in models}

last = ctx_s[idx, -1]
analytic = last[:, 1:2] + 0.75 * np.sin(last[:, 0:1]) + 0.15 * probe_actions  # (n_ctx, n_probe)

fig, axes = plt.subplots(1, len(models), figsize=(13, 4), squeeze=False)
for ax, (name, p) in zip(axes[0], probe_preds.items()):
    slopes = np.polyfit(probe_actions, p[:, :, 1].T, deg=1)[0]
    for i in range(8):
        ax.plot(probe_actions, p[i, :, 1], marker="o", ms=3, alpha=0.7)
        ax.plot(probe_actions, analytic[i], "k:", lw=1, alpha=0.5)
    ax.set(xlabel="kontrafaktische Aktion", ylabel=r"vorhergesagt $\dot\theta'$",
           title=f"{name}\nSteigung {slopes.mean():.3f} (wahr 0.15)")
fig.suptitle("Gepunktet schwarz: analytische Dynamik")
fig.tight_layout()

## Test 3: Kontextlängen-Sweep (Chronos; Baselines sind kontextfrei → Referenzlinien)

In [ ]:
context_lengths = [5, 10, 20, 50, 100, 150]
sweep_mse = []
for L in context_lengths:
    cs, ca, fa, fs = make_windows(states[40:], actions[40:], L=L, H=1, max_windows=500)
    p = chronos_model.predict(cs, ca, fa)
    sweep_mse.append(float(np.mean((p[:, 0] - fs[:, 0]) ** 2)))
    print(f"L={L}: MSE {sweep_mse[-1]:.5f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(context_lengths, sweep_mse, marker="o", label=chronos_model.name)
for m in (var_model, lin_model):
    ax.axhline(np.mean((preds_1step[m.name] - true_1step) ** 2), ls="--", lw=1, label=m.name)
ax.set(xscale="log", yscale="log", xlabel="Kontextlänge L", ylabel="Ein-Schritt-MSE")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

## Test 4: Mehrschritt-Rollout (open loop, L=64, H=50, Aktionen bekannt)

Chronos prognostiziert den Horizont direkt in einem Forward-Pass; VARX rekursiv,
Linear iteriert den Ein-Schritt-Fit.

In [ ]:
ctx_s, ctx_a, fut_a, fut_s = make_windows(states[40:], actions[40:], L=64, H=50, max_windows=300)
preds_rollout = {m.name: m.predict(ctx_s, ctx_a, fut_a) for m in models}

fig, ax = plt.subplots(figsize=(7, 4.5))
for name, p in preds_rollout.items():
    ax.plot(np.arange(1, 51), np.mean((p - fut_s) ** 2, axis=(0, 2)), label=name)
ax.set(yscale="log", xlabel="Horizont h (Schritte)", ylabel="MSE(h)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for j, ax in enumerate(axes):
    ax.plot(fut_s[j, :, 0], "k-", lw=2, label="wahr")
    for name, p in preds_rollout.items():
        ax.plot(p[j, :, 0], "--", label=name)
    ax.set(xlabel="Horizont", ylabel=r"$\theta$", title=f"Beispiel-Rollout {j}")
axes[0].legend()
fig.tight_layout()